# M4 · RAG에 넣을 실제 문서 고르기

① SIF 사례와 ⑥이 연결한 실제 고용노동부 재해조사보고서, ⑧ KOSHA GUIDE를 구분해서 목록을 만든다. ⑥의 CSV 5행 자체는 보고서 본문이 아니어서 색인하지 않는다. 문서마다 고유 ID·출처 링크·기관·기준일·원본 파일을 남긴다.

In [ ]:
from pathlib import Path
import json,hashlib
import pandas as pd
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None,'SANUP-P 프로젝트 폴더에서 실행하세요.'
PERSONAL=ROOT/'data'/'personal'
CORPUS=PERSONAL/'corpus';CORPUS.mkdir(parents=True,exist_ok=True)
REPORTS=PERSONAL/'reports';REPORTS.mkdir(parents=True,exist_ok=True)

## 1. ① SIF 사례를 문서로 묶기

In [ ]:
cases_path=PERSONAL/'processed'/'source_01_sif_cases.parquet'
assert cases_path.exists(),'M2 SIF 전처리를 먼저 실행하세요.'
cases=pd.read_parquet(cases_path)
assert len(cases)==6032 and cases.record_id.is_unique
def shown(value):
    return '' if pd.isna(value) else str(value).strip()
def case_text(row):
    fields=[('업종',row.industry_major),('작업',row.work_name),('고위험작업',row.hazard_work),
            ('재해종류',row.accident_type),('기인물',row.causal_object),
            ('재해개요',row.incident_summary),('재해유발요인',row.trigger_factor),
            ('위험성 감소대책',row.risk_reduction_measures if row.risk_measures_available else '')]
    return '\n'.join(f'{label}: {shown(value)}' for label,value in fields if shown(value))
documents=[]
for row in cases.itertuples(index=False):
    documents.append({'doc_id':str(row.record_id),'source_id':'01','source_type':'sif',
      'title':f'SIF 사례 {row.record_id}','organization':'한국산업안전보건공단',
      'source_url':'https://www.data.go.kr/data/15140383/fileData.do',
      'published_at':'','license':'공공저작물 제3유형: 출처표시·변경금지',
      'local_path':'','content':case_text(row),'industry_major':shown(row.industry_major),
      'equipment':shown(row.causal_object)})
print('① SIF 문서',len(documents))

## 2. ⑥ 실제 조사보고서 PDF 확인

In [ ]:
moel_dir=PERSONAL/'raw'/'source_06'/'reports'
moel_manifest=pd.read_csv(moel_dir/'manifest.csv',encoding='utf-8-sig')
titles={'20260501030':'대구 달서구 화물자동차 떨어짐 사고',
        '20260501028':'대구 달성군 이동식비계 떨어짐 사고',
        '20260501026':'경기 화성 톤백 깔림 사고'}
for row in moel_manifest.itertuples(index=False):
    path=moel_dir/row.filename
    assert hashlib.sha256(path.read_bytes()).hexdigest().upper()==row.sha256
    sequence=row.doc_id.split('-')[1]
    documents.append({'doc_id':row.doc_id,'source_id':'06','source_type':'moel_report',
      'title':titles.get(sequence,f'재해조사보고서 {sequence}'),
      'organization':'고용노동부','source_url':row.source_page,
      'published_at':'2026-05-26','license':'공공저작물 제3유형: 출처표시·변경금지',
      'local_path':str(path.relative_to(ROOT)),'content':'','industry_major':'','equipment':''})
print('⑥ 실제 PDF',len(moel_manifest),'건 (URL 목록 5행과 구분)')

## 3. ⑧ KOSHA GUIDE PDF와 색인 대조

In [ ]:
guide_dir=PERSONAL/'raw'/'source_08'
guides=pd.read_csv(guide_dir/'kosha_guide_index.csv',encoding='utf-8-sig').fillna('')
assert len(guides)==18 and guides.guide_id.is_unique
for row in guides.itertuples(index=False):
    path=guide_dir/'pdf'/f'{row.guide_id}.pdf'
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest().lower()==row.sha256.lower()
    assert str(row.source_url).startswith('https://portal.kosha.or.kr/')
    documents.append({'doc_id':row.guide_id,'source_id':'08','source_type':'kosha_guide',
      'title':row.title,'organization':'한국산업안전보건공단',
      'source_url':row.source_url,'published_at':row.publication_date,
      'license':'공개·가공 이용조건 추가 확인 필요',
      'local_path':str(path.relative_to(ROOT)),'content':'',
      'industry_major':'','equipment':row.equipment})
assert len(documents)==6053 and len({d['doc_id'] for d in documents})==len(documents)
manifest=pd.DataFrame([{k:v for k,v in d.items() if k!='content'} for d in documents])
manifest_path=CORPUS/'document_manifest.csv'
manifest.to_csv(manifest_path,index=False,encoding='utf-8-sig')
docs_path=CORPUS/'documents.jsonl'
with docs_path.open('w',encoding='utf-8') as stream:
    for doc in documents: stream.write(json.dumps(doc,ensure_ascii=False)+'\n')
display(manifest.groupby(['source_id','source_type']).size().rename('문서 수').to_frame())
print('문서 목록',manifest_path,'본문 참조',docs_path)

## 자료 구성 및 출처

- 문서 목록은 ① SIF 사례 6,032건, ⑥ 고용노동부 사고조사보고서 3건, ⑧ KOSHA GUIDE 18건으로 총 6,053개 문서다. 각 문서에 문서 ID, 제목, 기관, 원문 URL, 자료 유형 등 검색·출처 확인용 메타데이터를 연결했다.
- PDF 원본과 ZIP 내부 PDF는 파일 해시·목록을 기준으로 대조했다. ⑧의 18개 가이드는 목록과 원본 파일이 대응하고, ⑥ 보고서 3개도 인덱스에서 출처를 역추적할 수 있다.
- ⑦ 용어사전은 ⑧개 문서 코퍼스에 들어가는 안전 근거가 아니라 M7 검색어 확장 실험용 자료다. ②~⑤ 통계도 RAG 문서로 중복 적재하지 않고 대시보드에서 쓴다.
- ①·⑥은 출처표시·변경금지 조건의 공공저작물로 표시했다. 내부 개인 검증과 별개로 변형·공개·팀 공유 가능한 범위는 확정되지 않았으므로 이용 문의 초안과 별도 확인을 유지한다.

### 다음 단계와 주의사항

- 원본 파일은 수정하지 않고 OCR·정제본에는 원본 URL, 페이지, 추출 방식, OCR 재확인 표시를 남긴다.
- 공식 페이지는 로컬 수집 뒤 수정됐지만, `_20260401` 최신 파일을 다시 내려받아 로컬 원본과 SHA-256이 일치하는 것을 확인했다. 날짜 문자열과 메타데이터 기준의 차이는 별도 주의사항으로 남긴다.
- 문서가 폴더에 있다는 것과 이용 허락이 확인됐다는 것은 다르다. 외부 공유·발표자료 원문 인용 전 이용 조건을 확인한다.
